# Final Capstone Crisis Triage Notebook

This notebook reproduces the final testing workflow for the MSAI 699 capstone project.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.append(str(Path.cwd().parent / "src"))
from modeling import load_data, leave_one_event_out

In [ ]:
DATA_PATH = Path("../data/crisislex_t26_baseline_subset.csv")
df = load_data(DATA_PATH)
df.shape, df["event"].nunique(), df["binary_label"].value_counts().to_dict()

## Leave-one-event-out cross-validation
Each fold trains on four full crisis events and tests on the fifth unseen event.

In [ ]:
results = leave_one_event_out(df)
results.head()

In [ ]:
pooled = []
for model, g in results.groupby("Model"):
    tn, fp, fn, tp = g[["TN","FP","FN","TP"]].sum()
    precision = tp/(tp+fp) if (tp+fp) else 0
    recall = tp/(tp+fn) if (tp+fn) else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    acc = (tp+tn)/(tp+tn+fp+fn)
    pooled.append({"Model":model,"Accuracy":acc,"Precision":precision,"Recall":recall,"F1-score":f1,"False negatives":fn,"False positives":fp})
pd.DataFrame(pooled)

## Interpretation
The reliability policy that escalates a message if either model predicts it as informative improves recall and reduces missed informative reports at the cost of additional human-review workload.